# 04 · Development evaluation and reporting

**Inputs:** existing native checkpoints/completion receipts, frozen protocol/policy,
and verified Raw/Adaptive development datasets (or valid saved evaluation results).
**Outputs:** the 3×2 development matrix, per-class metrics, native plots/errors,
stage timing and provenance under `runs/development-v1/`.

Set paths below, then **Restart & Run All** in a fresh kernel. This phase never imports
the training runner or calls data initialization/preprocessing research. By default
it requires saved results. Set `ALLOW_INFERENCE=True` once to fill missing evaluations;
valid results are still reused. Editing plots never requires new inference.
Stale inputs/caches fail instead of being relabeled; use a new version/root.

Development validation is the exploratory mode. Final holdout execution is unavailable
here until the separate explicit release/model/protocol lock has been implemented.
The original training-code identity stays in each checkpoint's receipt.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from vehicle_detection.adaptive import validate_policy
from vehicle_detection.common import read_json, require_paths
from vehicle_detection.evaluation import completed_checkpoints, evaluate_development
from vehicle_detection.offline import verify_alignment
from vehicle_detection.protocol import validate_protocol

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
POLICY_PATH = ROOT / "runs/preprocessing-v1/policy.json"
TRAIN_ROOT = ROOT / "runs/training-s10-v1"
PRETRAINED = ROOT / "weights/yolo26n.pt"
INPUTS = {
    "raw": ROOT / "runs/data-v1/datasets/raw",
    "adaptive": ROOT / "runs/preprocessing-v1/development/adaptive",
}
OUTPUT = ROOT / "runs/development-v1"
ALLOW_INFERENCE = False
require_paths(
    protocol=PROTOCOL_PATH,
    policy=POLICY_PATH,
    pretrained=PRETRAINED,
    training=TRAIN_ROOT / "pair.json",
    **{k: v / "verified.json" for k, v in INPUTS.items()},
)
protocol = validate_protocol(read_json(PROTOCOL_PATH))
policy = validate_policy(read_json(POLICY_PATH), protocol)
checkpoints = {"pretrained": PRETRAINED, **completed_checkpoints(TRAIN_ROOT, protocol)}
verify_alignment(list(INPUTS.values()), deep=False)

In [ ]:
results = {}
for model_name, checkpoint in checkpoints.items():
    for input_name, dataset in INPUTS.items():
        name = f"{model_name}-{input_name}"
        results[name] = evaluate_development(
            protocol,
            policy,
            dataset,
            checkpoint,
            OUTPUT / name,
            pretrained=model_name == "pretrained",
            allow_inference=ALLOW_INFERENCE,
        )
matrix = pd.DataFrame(
    [
        {"model": model, "input": treatment, **results[f"{model}-{treatment}"]["overall"]}
        for model in checkpoints
        for treatment in INPUTS
    ]
)
display(matrix)
per_class = pd.DataFrame(
    [{"evaluation": name, **row} for name, result in results.items() for row in result["classes"]]
)
display(per_class)
print("Native exported-file timing; excludes offline enhancement and application I/O:")
display(pd.DataFrame({name: r["speed_ms"] for name, r in results.items()}).T)

In [ ]:
matrix.pivot(index="model", columns="input", values="mAP50_95").plot.bar(
    figsize=(8, 3), rot=20, title="Native development mAP50:95 · six observed classes"
)
plt.tight_layout()
plt.show()
for name in results:
    display(Image(filename=str(OUTPUT / name / "confusion_matrix_normalized.png"), width=650))
    examples = sorted((OUTPUT / name).glob("val_batch*_pred.jpg"))
    if examples:
        display(Image(filename=str(examples[0]), width=900))
print("No final-test execution or model-selection claims from this exploratory matrix.")